### **Protein Structural Analysis Pipeline**


An elaborate pipeline for analyzing protein structures obtained from AlphaFold.:
1. Loading and RMSD-based Selection: Extracting AlphaFold models from a ZIP archive and identifying the most representative model.
2. Post-Translational Modification (PTM) with gemmi: Programmatically modifying specific residues (e.g., tryptophan oxidation).
3. Visualization: Interactive 3D visualization of the protein structures.
4. Solvent Accessible Surface Area (SASA) Analysis: Quantifying the surface accessibility of specific residues.
5. Proximity Analysis: Identifying residues in close spatial contact.
The results from SASA and Proximity analyses are saved to CSV files.

## 1. Initial Setup: Library Installation and Imports

In [ ]:
# Install necessary libraries
!pip install openmm py3Dmol gemmi freesasa biopython
!pip install openpyxl

# Core Python libraries
import zipfile
import os
import itertools
import tempfile
from collections import Counter
import warnings # Added for warning suppression

# Scientific computing libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Structural biology libraries
import gemmi
import freesasa
from Bio.PDB import MMCIFParser, Superimposer, PDBIO, MMCIFIO, Atom, NeighborSearch, Selection, PDBParser
from Bio.PDB.PDBParser import PDBConstructionWarning # Added for warning suppression

# Suppress specific Biopython warnings that are non-critical, as they don't affect results
warnings.filterwarnings('ignore', category=PDBConstructionWarning)

print("All necessary libraries installed and imported.")

## 2. Load AlphaFold Data and RMSD Analysis

This section extracts AlphaFold models from a ZIP file, calculates pairwise RMSD between models, and selects the model with the lowest RMSD as the 'best' representative for further analysis. This 'best' model is then saved as a CIF file.

In [ ]:
from Bio.PDB import MMCIFParser, Superimposer, PDBIO, MMCIFIO


# zip file path defined
alphafold_zip_file = '/content/fold_lrp8_vl_ptm_n_d (1).zip'
extraction_path = '/content/alphafold_models'
os.makedirs(extraction_path, exist_ok=True)

# --- Function to Extract Models ---
def extract_models(zip_path, extract_dir):
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_dir)
    print(f"Extracted models to: {extract_dir}")
    return [os.path.join(extract_dir, f) for f in os.listdir(extract_dir) if f.endswith(('.pdb', '.cif'))]

# --- Function to Read Structures (handling PDB and CIF) ---
def read_structure(file_path):
    parser = None
    if file_path.endswith('.pdb'):
        parser = PDBParser()
    elif file_path.endswith('.cif'):
        parser = MMCIFParser()
    else:
        raise ValueError(f"Unsupported file format for {file_path}")
    return parser.get_structure(os.path.basename(file_path), file_path)

# --- Function to Calculate RMSD ---
def calculate_rmsd(structure1, structure2):
    # Select atoms for superposition (e.g., C-alpha atoms)
    atoms1 = [atom for atom in structure1.get_atoms() if atom.get_name() == 'CA']
    atoms2 = [atom for atom in structure2.get_atoms() if atom.get_name() == 'CA']

    # Only superimpose if both have enough C-alpha atoms
    if len(atoms1) == 0 or len(atoms2) == 0:
        return float('inf') # Cannot calculate RMSD

    # Ensure they have the same number of atoms for comparison
    # This simple approach assumes structures are already aligned or have same length
    # For real-world use, you might need sequence alignment and mapping
    if len(atoms1) != len(atoms2):
        min_len = min(len(atoms1), len(atoms2))
        atoms1 = atoms1[:min_len]
        atoms2 = atoms2[:min_len]

    sup = Superimposer()
    sup.set_atoms(atoms1, atoms2)
    return sup.rms

# --- Main RMSD Analysis Logic ---
print(f"Starting RMSD analysis for models from {alphafold_zip_file}")

with tempfile.TemporaryDirectory() as temp_dir:
    extracted_model_paths = extract_models(alphafold_zip_file, temp_dir)
    if not extracted_model_paths:
        print(f"No models found in {alphafold_zip_file}. Please check the zip file content and path.")
    else:
        structures = {path: read_structure(path) for path in extracted_model_paths}
        model_names = list(structures.keys())

        rmsd_matrix = np.zeros((len(model_names), len(model_names)))

        for i in range(len(model_names)):
            for j in range(i, len(model_names)):
                rms = calculate_rmsd(structures[model_names[i]][0], structures[model_names[j]][0]) # Assume single model in structure obj
                rmsd_matrix[i, j] = rms
                rmsd_matrix[j, i] = rms

        rmsd_df = pd.DataFrame(rmsd_matrix, index=model_names, columns=model_names)
        print("Pairwise RMSD Matrix:")
        display(rmsd_df)

        # Select the 'best' model as the one with the lowest average RMSD to all other models
        if not rmsd_df.empty:
            average_rmsd = rmsd_df.mean(axis=1)
            print("\nAverage RMSD for each model:")
            display(average_rmsd)
            best_model_path = average_rmsd.idxmin()
            print(f"\nBest model (lowest average RMSD): {best_model_path}")

            # Save the best model as a CIF file, as other parts of the notebook expect CIF
            best_model_name = os.path.basename(best_model_path)
            output_cif_filename = f"best_model_from_rmsd_{os.path.splitext(best_model_name)[0]}.cif"

            # Biopython's MMCIFIO can write both PDB and CIF structures to CIF
            io = MMCIFIO()
            io.set_structure(structures[best_model_path])
            io.save(output_cif_filename)

            global cif_file_to_analyze_ptm
            cif_file_to_analyze_ptm = os.path.abspath(output_cif_filename)

            print(f"Best model saved as CIF to: {cif_file_to_analyze_ptm}")
        else:
            print("Could not determine best model as RMSD matrix is empty.")
            cif_file_to_analyze_ptm = None

    # You might want to remove the extracted directory if not using a TemporaryDirectory
    # For this example, TemporaryDirectory handles cleanup automatically.

## 3. Coordinate Modification (PTM) using `gemmi`

This section defines a function to add an oxygen atom to specific methionine residues to simulate oxidation. This modification is applied to the selected best model, and the modified structure is saved to a new CIF file.

In [ ]:
def add_oxygen_to_atom_gemmi(
    gemmi_structure,
    chain_id,
    res_id,
    target_res_name,
    target_atom_name,
    new_o_atom_name,
    reference_atom_names,
    bond_length=1.5
):
    """
    Adds an oxygen atom to a specified atom within a residue in a gemmi.Structure.
    The oxygen atom is placed along the vector pointing from the average position
    of reference atoms to the target_atom_name, extended outwards from target_atom_name.

    Args:
        gemmi_structure (gemmi.Structure): The structure object to modify.
        chain_id (str): The ID of the chain containing the target residue.
        res_id (int): The sequence ID of the target residue.
        target_res_name (str): The 3-letter name of the target residue (e.g., 'MET', 'TRP').
        target_atom_name (str): The name of the atom in the target residue where the new oxygen will be bonded (e.g., 'SD' for Methionine).
        new_o_atom_name (str): The desired name for the newly added oxygen atom (e.g., 'OXT').
        reference_atom_names (list of str): A list of atom names within the target residue that define the outward direction
                                             from the target_atom_name. The new oxygen will be placed along the vector
                                             pointing from the average position of these reference atoms to the target_atom_name.
        bond_length (float, optional): The desired bond length for the new bond in Angstroms (default: 1.5 Å).

    Returns:
        bool: True if the oxygen atom was successfully added, False otherwise.
    """
    target_atom = None
    reference_coords = []

    for model in gemmi_structure:
        for chain in model:
            if chain.name == chain_id:
                for res in chain:
                    if res.name == target_res_name and res.seqid.num == res_id:
                        # Debugging: Print all atom names in the target residue
                        print(f"Found {target_res_name}{res_id} in chain {chain_id}. Atoms: {[atom.name for atom in res]}")
                        for atom in res:
                            # For AlphaFold models, sometimes atom names are padded with spaces.
                            # Stripping spaces ensures a robust match.
                            if atom.name.strip() == target_atom_name:
                                target_atom = atom
                            if atom.name.strip() in reference_atom_names:
                                reference_coords.append(np.array([atom.pos.x, atom.pos.y, atom.pos.z]))


    if target_atom is None:
        print(f"Warning: Target atom '{target_atom_name}' not found for {target_res_name}{res_id} in chain {chain_id}.")
        return False

    if not reference_coords:
        print(f"Warning: Reference atoms {reference_atom_names} not found for {target_res_name}{res_id} in chain {chain_id}. Cannot determine direction.")
        return False

    avg_reference_coord = np.mean(reference_coords, axis=0)
    target_atom_coord = np.array([target_atom.pos.x, target_atom.pos.y, target_atom.pos.z])

    # Vector from average reference point to target atom
    direction_vector = target_atom_coord - avg_reference_coord
    norm_direction_vector = direction_vector / np.linalg.norm(direction_vector)

    # Calculate new oxygen position
    new_o_coord = target_atom_coord + norm_direction_vector * bond_length

    # Create new oxygen atom
    new_o_atom = gemmi.Atom()
    new_o_atom.name = new_o_atom_name
    new_o_atom.element = gemmi.Element('O')
    new_o_atom.pos.x = new_o_coord[0]
    new_o_atom.pos.y = new_o_coord[1]
    new_o_atom.pos.z = new_o_coord[2]
    new_o_atom.occ = target_atom.occ
    new_o_atom.b_iso = target_atom.b_iso

    # Add the new atom to the residue
    for model in gemmi_structure:
        for chain in model:
            if chain.name == chain_id:
                for res in chain:
                    if res.name == target_res_name and res.seqid.num == res_id:
                        res.add_atom(new_o_atom)
                        print(f"Added atom {new_o_atom_name} to {target_res_name}{res_id} in chain {chain_id}.")
                        return True
    return False

# --- Perform modification ---
# Ensure correct file is used from RMSD analysis
input_cif_for_ptm = '/content/best_model_from_rmsd_fold_lrp8_vl_ptm_n_d_model_2.cif' # This is the best model from previous RMSD analysis
modified_structure_file = os.path.abspath('best_model_tryptophan_oxidized.cif') # Output filename for modified structure

tryptophan_targets = [47] # Target Tryptophan 47
chain_id_ptm = 'A'
target_res_name_ptm = 'TRP' # Targeting Tryptophan
target_atom_name_ptm = 'CH2' # Targeting the CH2 atom in the indole ring
new_o_atom_name = 'OXT_IND' # New oxygen atom name, e.g., for indole oxidation
reference_atom_names_ptm = ['CZ3'] # Use CZ3 to define direction for CH2 in Tryptophan's indole ring

print("Starting modification process...")

# Load the structure once using gemmi
doc = gemmi.cif.read_file(input_cif_for_ptm)
modified_gemmi_structure = gemmi.make_structure_from_block(doc.sole_block())

modified_res = []
for res_id in tryptophan_targets:
    success = add_oxygen_to_atom_gemmi(
        modified_gemmi_structure,
        chain_id_ptm,
        res_id,
        target_res_name_ptm,
        target_atom_name_ptm,
        new_o_atom_name,
        reference_atom_names_ptm
    )
    if success:
        modified_res.append((chain_id_ptm, res_id))

# Save the modified structure
modified_gemmi_structure.make_mmcif_document().write_file(modified_structure_file)

if modified_res:
    print(f"\nSuccessfully added oxygen to Tryptophan residues: {modified_res}. Modified structure saved to {modified_structure_file}")
elif tryptophan_targets:
    # Only print this if there were target residues, but none were modified (e.g., not found)
    print("No Tryptophan residues were modified. Check if residues exist and target atoms are correct.")
else:
    print("No Tryptophan residues were targeted for modification.")

## 4. Visualization of Best Model with PTM using `py3Dmol`

This section visualizes the original best model with target residues highlighted, and then the modified structure with the added oxygen atoms clearly marked, along with other highlighted residues for context.

In [ ]:
import py3Dmol

if os.path.exists(modified_structure_file):
    with open(modified_structure_file, 'r') as f:
        modified_cif_content = f.read()

    view_modified = py3Dmol.view(width=800, height=600)
    view_modified.addModel(modified_cif_content, 'cif')

    # Style the entire protein in grey cartoon/ribbon
    view_modified.setStyle({'cartoon': {'color': 'grey'}})

    # Highlight the modified Tryptophan 47
    chain_id_viz = 'A'
    tryptophan_targets_viz = [47]
    new_o_atom_name_viz = 'OXT_IND'

    for res_id in tryptophan_targets_viz:
        # Style the entire Tryptophan residue in magenta
        view_modified.addStyle({'resi': str(res_id), 'chain': chain_id_viz}, {'stick': {'radius': 0.3, 'color': 'magenta'}})
        view_modified.addStyle({'resi': str(res_id), 'chain': chain_id_viz}, {'sphere': {'radius': 0.5, 'color': 'magenta'}})

        # Explicitly color the newly added oxygen (OXT_IND) in yellow and make it more prominent
        view_modified.addStyle({'resi': str(res_id), 'chain': chain_id_viz, 'atom': new_o_atom_name_viz}, {'sphere': {'radius': 0.7, 'color': 'yellow'}})
        view_modified.addStyle({'resi': str(res_id), 'chain': chain_id_viz, 'atom': new_o_atom_name_viz}, {'stick': {'radius': 0.3, 'color': 'yellow'}})

    # Add highlighting for Aspartic Acid 33 in blue sticks and red balls
    asp_res_id = 33
    view_modified.addStyle({'resi': str(asp_res_id), 'chain': chain_id_viz}, {'stick': {'radius': 0.3, 'color': 'blue'}})
    view_modified.addStyle({'resi': str(asp_res_id), 'chain': chain_id_viz}, {'sphere': {'radius': 0.5, 'color': 'red'}})

    view_modified.zoomTo();
    print("\nModified model with oxidized Tryptophan 47 (entire residue in magenta, added oxygen in yellow) and Aspartic Acid 33 (blue sticks, red balls) highlighted:")
    view_modified.show()
else:
    print("Visualization skipped: Modified CIF file not found.")

### Selecting the native strucutre with no PTMs.

In [ ]:
from Bio.PDB import MMCIFParser, Superimposer, PDBIO, MMCIFIO


# zip file path defined - UPDATED for native structure
alphafold_zip_file = '/content/fold_lrp8_vh_vl_native.zip' # User-specified native zip file
extraction_path = '/content/alphafold_native_models'
os.makedirs(extraction_path, exist_ok=True)

# --- Function to Extract Models ---
def extract_models(zip_path, extract_dir):
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_dir)
    print(f"Extracted models to: {extract_dir}")
    return [os.path.join(extract_dir, f) for f in os.listdir(extract_dir) if f.endswith(('.pdb', '.cif'))]

# --- Function to Read Structures (handling PDB and CIF) ---
def read_structure(file_path):
    parser = None
    if file_path.endswith('.pdb'):
        parser = PDBParser()
    elif file_path.endswith('.cif'):
        parser = MMCIFParser()
    else:
        raise ValueError(f"Unsupported file format for {file_path}")
    return parser.get_structure(os.path.basename(file_path), file_path)

# --- Function to Calculate RMSD ---
def calculate_rmsd(structure1, structure2):
    # Select atoms for superposition (e.g., C-alpha atoms)
    atoms1 = [atom for atom in structure1.get_atoms() if atom.get_name() == 'CA']
    atoms2 = [atom for atom in structure2.get_atoms() if atom.get_name() == 'CA']

    # Only superimpose if both have enough C-alpha atoms
    if len(atoms1) == 0 or len(atoms2) == 0:
        return float('inf') # Cannot calculate RMSD

    # Ensure they have the same number of atoms for comparison
    # This simple approach assumes structures are already aligned or have same length
    # For real-world use, you might need sequence alignment and mapping
    if len(atoms1) != len(atoms2):
        min_len = min(len(atoms1), len(atoms2))
        atoms1 = atoms1[:min_len]
        atoms2 = atoms2[:min_len]

    sup = Superimposer()
    sup.set_atoms(atoms1, atoms2)
    return sup.rms

# --- Main RMSD Analysis Logic ---
print(f"Starting RMSD analysis for models from {alphafold_zip_file}")

with tempfile.TemporaryDirectory() as temp_dir:
    extracted_model_paths = extract_models(alphafold_zip_file, temp_dir)
    if not extracted_model_paths:
        print(f"No models found in {alphafold_zip_file}. Please check the zip file content and path.")
    else:
        structures = {path: read_structure(path) for path in extracted_model_paths}
        model_names = list(structures.keys())

        rmsd_matrix = np.zeros((len(model_names), len(model_names)))

        for i in range(len(model_names)):
            for j in range(i, len(model_names)):
                rms = calculate_rmsd(structures[model_names[i]][0], structures[model_names[j]][0]) # Assume single model in structure obj
                rmsd_matrix[i, j] = rms
                rmsd_matrix[j, i] = rms

        rmsd_df = pd.DataFrame(rmsd_matrix, index=model_names, columns=model_names)
        print("Pairwise RMSD Matrix:")
        display(rmsd_df)

        # Select the 'best' model as the one with the lowest average RMSD to all other models
        if not rmsd_df.empty:
            average_rmsd = rmsd_df.mean(axis=1)
            print("\nAverage RMSD for each model:")
            display(average_rmsd)
            best_model_path = average_rmsd.idxmin()
            print(f"\nBest model (lowest average RMSD): {best_model_path}")

            # Save the best model as a CIF file, as other parts of the notebook expect CIF
            best_model_name = os.path.basename(best_model_path)
            # Rename output CIF to clearly distinguish it as the native best model
            output_cif_filename = f"best_native_model_from_rmsd_{os.path.splitext(best_model_name)[0]}.cif"

            # Biopython's MMCIFIO can write both PDB and CIF structures to CIF
            io = MMCIFIO()
            io.set_structure(structures[best_model_path])
            io.save(output_cif_filename)

            global cif_file_to_analyze_ptm
            cif_file_to_analyze_ptm = os.path.abspath(output_cif_filename)

            print(f"Best native model saved as CIF to: {cif_file_to_analyze_ptm}")
        else:
            print("Could not determine best model as RMSD matrix is empty.")
            cif_file_to_analyze_ptm = None

    # You might want to remove the extracted directory if not using a TemporaryDirectory
    # For this example, TemporaryDirectory handles cleanup automatically.

## 7. Visualization of Best Native Model using `py3Dmol`

In [ ]:
import py3Dmol
import os # Ensure os module is imported for os.path.exists and os.path.basename

# Explicitly set the file path as requested by the user
cif_file_to_analyze_ptm = '/content/best_native_model_from_rmsd_fold_lrp8_vh_vl_native_model_1.cif'

# The variable cif_file_to_analyze_ptm was updated by the previous RMSD analysis cell
# to point to the best native model.
if os.path.exists(cif_file_to_analyze_ptm):
    with open(cif_file_to_analyze_ptm, 'r') as f:
        native_cif_content = f.read()

    view_native = py3Dmol.view(width=800, height=600);
    view_native.addModel(native_cif_content, 'cif');

    # Set a default grey cartoon style for the entire protein
    view_native.setStyle({'cartoon': {'color': 'lightgrey'}})

    # Highlight Tryptophan 47 in magenta ball and stick
    # Assuming TRP 47 is in chain 'A' based on previous context
    view_native.addStyle({'resi': '47', 'resn': 'TRP', 'chain': 'A'}, {'stick': {'radius': 0.3, 'color': 'magenta'}})
    view_native.addStyle({'resi': '47', 'resn': 'TRP', 'chain': 'A'}, {'sphere': {'radius': 0.5, 'color': 'magenta'}})

    # Highlight Asparagine 174 in blue ball and stick
    # Assuming ASN 174 is in chain 'A' based on previous context
    view_native.addStyle({'resi': '174', 'resn': 'ASN', 'chain': 'A'}, {'stick': {'radius': 0.3, 'color': 'blue'}})
    view_native.addStyle({'resi': '174', 'resn': 'ASN', 'chain': 'A'}, {'sphere': {'radius': 0.5, 'color': 'blue'}})

    view_native.zoomTo();
    print(f"\nBest native model ({os.path.basename(cif_file_to_analyze_ptm)}) displayed with updated custom coloring:")
    view_native.show()
else:
    print(f"Visualization skipped: Native CIF file not found at {cif_file_to_analyze_ptm}.")

## 5. Solvent Accessible Surface Area (SASA) Analysis

This section defines a robust function to calculate the SASA for specified residues. It then applies this function to both the native (unmodified) and modified protein structures, and saves the results to CSV files.

### Calculate SASA of Native and PTM strcutureS




In [ ]:
def get_residue_sasa(cif_file_path, chain_id, res_id, atom_names=None):
    """
    Calculates the SASA for a specific residue (or specific atoms within it) in a CIF file.
    Handles CIF to PDB conversion for freesasa compatibility.
    """
    temp_pdb_path = None
    structure_path_for_freesasa = cif_file_path

    try:
        # Try to load with freesasa directly from CIF
        structure = freesasa.Structure(cif_file_path)
    except Exception as e:
        # print(f"Freesasa direct CIF load failed: {e}. Attempting conversion...") # Suppress verbose error
        # Fallback 1: Try Biopython to convert CIF to PDB
        try:
            parser = MMCIFParser()
            biopython_structure = parser.get_structure('tmp', cif_file_path)

            with tempfile.NamedTemporaryFile(suffix=".pdb", delete=False) as tmp_pdb:
                temp_pdb_path = tmp_pdb.name
                io = PDBIO()
                io.set_structure(biopython_structure)
                io.save(temp_pdb_path)

            structure = freesasa.Structure(temp_pdb_path)
            structure_path_for_freesasa = temp_pdb_path
            # print(f"Successfully converted CIF to PDB using Biopython for freesasa: {temp_pdb_path}") # Suppress verbose output

        except Exception as e_bio:
            # print(f"Biopython conversion failed: {e_bio}. Attempting gemmi conversion...") # Suppress verbose error
            # Fallback 2: Try gemmi to convert CIF to PDB
            try:
                doc = gemmi.cif.read_file(cif_file_path)
                gemmi_block = doc.sole_block()
                gemmi_structure = gemmi.make_structure_from_block(gemmi_block)
                with tempfile.NamedTemporaryFile(suffix=".pdb", delete=False) as tmp_pdb:
                    temp_pdb_path = tmp_pdb.name
                    gemmi_structure.write_pdb(temp_pdb_path)
                structure = freesasa.Structure(temp_pdb_path)
                structure_path_for_freesasa = temp_pdb_path
                # print(f"Successfully converted CIF to PDB using gemmi for freesasa: {temp_pdb_path}") # Suppress verbose output

            except Exception as e_gemmi:
                print(f"Gemmi conversion also failed: {e_gemmi}. Cannot calculate SASA for {cif_file_path}.")
                if temp_pdb_path and os.path.exists(temp_pdb_path): os.unlink(temp_pdb_path)
                return 0.0 # Return 0.0 if SASA calculation fails

    # Perform SASA calculation
    result = freesasa.calc(structure)
    total_sasa = 0.0
    found_atoms_count = 0

    # Iterate through atoms in the freesasa structure to calculate SASA
    for i in range(structure.nAtoms()):
        atom_chain_label = structure.chainLabel(i)
        atom_residue_number = structure.residueNumber(i).strip()
        atom_name = structure.atomName(i)

        if atom_chain_label == chain_id and atom_residue_number == str(res_id):
            if atom_names is None or atom_name in atom_names:
                total_sasa += result.atomArea(i)
                found_atoms_count += 1

    if temp_pdb_path and os.path.exists(temp_pdb_path):
        os.unlink(temp_pdb_path)
        # print(f"Cleaned up temporary PDB file: {temp_pdb_path}") # Suppress verbose output

    if found_atoms_count == 0:
        # print(f"Warning: No matching atoms found for Chain {chain_id}, Residue {res_id}" + \
        #       (f" with atom names {atom_names}" if atom_names else "") + f" in {cif_file_path}.")
        return 0.0

    return total_sasa

# Define the file paths as requested by the user
native_structure_for_sasa = '/content/best_native_model_from_rmsd_fold_lrp8_vh_vl_native_model_1.cif'
modified_structure_for_sasa = '/content/best_model_tryptophan_oxidized.cif'

# Residues to analyze for SASA
sasa_targets_native = [
    {'name': 'Tryptophan', 'res_id': 47, 'chain': 'A'},
    {'name': 'Asparagine', 'res_id': 33, 'chain': 'B'}
]

sasa_targets_modified = [
    {'name': 'Tryptophan', 'res_id': 47, 'chain': 'A'},
    {'name': 'Aspartic Acid', 'res_id': 33, 'chain': 'B'}
]


# List to store all SASA results
all_sasa_results = []

print("Calculating SASA for Native Structure...")
for res_info in sasa_targets_native:
    res_name = res_info['name']
    res_id = res_info['res_id']
    chain_id = res_info['chain']
    sasa_value = get_residue_sasa(native_structure_for_sasa, chain_id, res_id)
    all_sasa_results.append({
        'Residue_Name': res_name,
        'Residue_ID': res_id,
        'Chain_ID': chain_id,
        'Structure_Type': 'Native',
        'SASA_Angstrom2': sasa_value
    })
    print(f"  SASA for {res_name} {res_id} (Chain {chain_id}) in Native Structure: {sasa_value:.2f} Å²")

print("\nCalculating SASA for Modified (PTM) Structure...")
for res_info in sasa_targets_modified:
    res_name = res_info['name']
    res_id = res_info['res_id']
    chain_id = res_info['chain']
    sasa_value = get_residue_sasa(modified_structure_for_sasa, chain_id, res_id)
    all_sasa_results.append({
        'Residue_Name': res_name,
        'Residue_ID': res_id,
        'Chain_ID': chain_id,
        'Structure_Type': 'PTM',
        'SASA_Angstrom2': sasa_value
    })
    print(f"  SASA for {res_name} {res_id} (Chain {chain_id}) in Modified Structure: {sasa_value:.2f} Å²")

# Create DataFrame and save to CSV
sasa_df = pd.DataFrame(all_sasa_results)
output_combined_csv_filename_sasa = 'combined_sasa_analysis_results.csv'
sasa_df.to_csv(output_combined_csv_filename_sasa, index=False)

print(f"\nCombined Native and PTM SASA analysis results saved to {output_combined_csv_filename_sasa}")
print("SASA Analysis Results:")
display(sasa_df)

## 6. Proximity Analysis

This section defines a function to find nearest neighbors for specific residues within a cutoff distance. It then performs this analysis on both the native and modified protein structures, and saves the results to CSV files.

In [ ]:
import warnings
# Suppress specific Biopython warnings that are non-critical, as they don't affect results
warnings.filterwarnings('ignore', category=PDBConstructionWarning)

def find_nearest_neighbors_for_residue(cif_file_path, target_chain_id, target_res_id, cutoff_distance=4.0):
    """
    Finds all atoms from other residues within a specified cutoff distance of any atom
    in the target residue, using Biopython's NeighborSearch.

    Args:
        cif_file_path (str): Path to the CIF file.
        target_chain_id (str): The ID of the target residue's chain.
        target_res_id (int): The sequence ID of the target residue.
        cutoff_distance (float): The maximum distance (in Angstroms) to search for neighbors.

    Returns:
        list: A list of dictionaries, where each dictionary represents a neighboring atom
              and includes its details (atom_name, res_name, res_id, chain_id, distance).
    """
    temp_pdb_path = None
    structure = None
    try:
        # Attempt to parse directly with Biopython's MMCIFParser
        parser = MMCIFParser()
        structure = parser.get_structure('protein', cif_file_path)
    except Exception as e:
        # print(f"Biopython MMCIFParser failed: {e}. Attempting gemmi conversion to PDB...") # Suppress verbose error
        try:
            # Fallback: use gemmi to read CIF and write to a temporary PDB file
            doc = gemmi.cif.read_file(cif_file_path)
            gemmi_block = doc.sole_block()
            gemmi_structure = gemmi.make_structure_from_block(gemmi_block)
            with tempfile.NamedTemporaryFile(suffix=".pdb", delete=False) as tmp_pdb:
                temp_pdb_path = tmp_pdb.name
                gemmi_structure.write_pdb(temp_pdb_path)

            # Now parse the temporary PDB file with Biopython's PDBParser
            parser = PDBParser()
            structure = parser.get_structure('protein_pdb', temp_pdb_path)
            # print(f"Successfully converted CIF to PDB using gemmi for Biopython parsing: {temp_pdb_path}") # Suppress verbose output
        except Exception as e_gemmi_pdb:
            print(f"Gemmi conversion to PDB and Biopython PDBParser also failed: {e_gemmi_pdb}. Cannot perform proximity analysis for {cif_file_path}.")
            return []
    finally:
        # Clean up temporary PDB file if created
        if temp_pdb_path and os.path.exists(temp_pdb_path):
            os.unlink(temp_pdb_path)

    if structure is None:
        return []

    # Assuming single model for AlphaFold outputs, or just taking the first one
    model = structure[0]

    target_residue = None
    for chain in model:
        if chain.id == target_chain_id:
            for residue in chain:
                # Residue IDs in Biopython are tuples (hetatm_flag, res_sequence_id, insertion_code)
                # We compare the res_sequence_id (second element of the tuple)
                if residue.id[1] == target_res_id:
                    target_residue = residue
                    break
            if target_residue:
                break

    if not target_residue:
        print(f"Target residue {target_res_id} (Chain {target_chain_id}) not found using Biopython.")
        return []

    # Get all atoms in the structure to build NeighborSearch
    all_atoms = Selection.unfold_entities(structure, 'A') # 'A' for atoms

    # Initialize NeighborSearch
    ns = NeighborSearch(all_atoms)

    nearby_atoms_info = []
    for target_atom in target_residue.get_atoms():
        # Find neighbors for each atom in the target residue
        neighbors = ns.search(target_atom.coord, cutoff_distance)

        for neighbor_atom in neighbors:
            # Exclude atoms from the target residue itself
            if neighbor_atom.get_parent().id == target_residue.id and \
               neighbor_atom.get_parent().get_parent().id == target_chain_id:
                continue

            # Calculate distance using Biopython's overloaded operator for Atom objects
            distance = target_atom - neighbor_atom

            nearby_atoms_info.append({
                'query_atom': target_atom.get_id(),
                'neighbor_atom_name': neighbor_atom.get_id(),
                'neighbor_res_name': neighbor_atom.get_parent().get_resname(),
                'neighbor_res_id': neighbor_atom.get_parent().get_id()[1], # Residue sequence ID
                'neighbor_chain_id': neighbor_atom.get_parent().get_parent().get_id(),
                'distance': distance
            })

    # Remove duplicate neighbors (if multiple atoms of the target residue find the same neighbor atom)
    unique_nearby_atoms = {}
    for info in nearby_atoms_info:
        # Key includes query_atom, chain, res_id, and atom_name of the neighbor
        key = (info['query_atom'], info['neighbor_chain_id'], info['neighbor_res_id'], info['neighbor_atom_name'])
        if key not in unique_nearby_atoms or info['distance'] < unique_nearby_atoms[key]['distance']:
            unique_nearby_atoms[key] = info

    sorted_results = sorted(unique_nearby_atoms.values(), key=lambda x: x['distance'])
    return sorted_results

# --- Proximity analysis on the Native structure ---
print("Performing Proximity Analysis on Native Structure...")
native_structure_for_proximity = '/content/best_native_model_from_rmsd_fold_lrp8_vh_vl_native_model_1.cif' # Explicitly set native structure file as per user

# Define the residues to analyze for the NATIVE structure
native_residues_to_analyze_proximity = [
    {'res_name': 'Tryptophan', 'res_id': 47, 'chain': 'A'},
    {'res_name': 'Asparagine', 'res_id': 33, 'chain': 'B'}
]

cutoff = 4.0 # Angstroms

all_proximity_results_native = []

for res_info in native_residues_to_analyze_proximity:
    res_name_str = res_info['res_name']
    res_id_int = res_info['res_id']
    target_chain_proximity = res_info['chain']

    nearest_neighbors_native = find_nearest_neighbors_for_residue(native_structure_for_proximity, target_chain_proximity, res_id_int, cutoff)

    if nearest_neighbors_native:
        for neighbor in nearest_neighbors_native:
            all_proximity_results_native.append({
                'Target_Residue': f"{res_name_str} {res_id_int}",
                'Target_Chain': target_chain_proximity,
                'Query_Atom': neighbor['query_atom'],
                'Neighbor_Residue': f"{neighbor['neighbor_res_name']} {neighbor['neighbor_res_id']}",
                'Neighbor_Chain': neighbor['neighbor_chain_id'],
                'Neighbor_Atom': neighbor['neighbor_atom_name'],
                'Distance_Angstroms': neighbor['distance']
            })
    else:
        print(f"  No nearest neighboring atoms found within {cutoff} Å of {res_name_str} {res_id_int} (Chain {target_chain_proximity}) in NATIVE structure.")

# Create and display DataFrame
if all_proximity_results_native:
    proximity_df_native = pd.DataFrame(all_proximity_results_native)
    print("\n--- Proximity Analysis Results DataFrame for Native Structure ---")
    display(proximity_df_native)
    # Save to CSV
    proximity_output_csv_filename_native = 'proximity_analysis_results_native_new_targets.csv'
    proximity_df_native.to_csv(proximity_output_csv_filename_native, index=False)
    print(f"Proximity analysis results for native structure saved to {proximity_output_csv_filename_native}")
else:
    print("\nNo proximity results to display for the native structure.")


# --- Proximity analysis on the Modified structure ---
print("\nPerforming Proximity Analysis on Modified PTM Structure...")
modified_ptm_structure_for_proximity = '/content/best_model_tryptophan_oxidized.cif' # Explicitly set PTM structure file as per user

# Define the residues to analyze for the MODIFIED PTM structure
modified_residues_to_analyze_proximity = [
    {'res_name': 'Tryptophan', 'res_id': 47, 'chain': 'A'},
    {'res_name': 'Aspartic Acid', 'res_id': 33, 'chain': 'B'}
]

all_proximity_results_modified = []

for res_info in modified_residues_to_analyze_proximity:
    res_name_str = res_info['res_name']
    res_id_int = res_info['res_id']
    target_chain_proximity = res_info['chain']

    nearest_neighbors_modified = find_nearest_neighbors_for_residue(
        modified_ptm_structure_for_proximity, target_chain_proximity, res_id_int, cutoff
    )

    if nearest_neighbors_modified:
        for neighbor in nearest_neighbors_modified:
            all_proximity_results_modified.append({
                'Target_Residue': f"{res_name_str} {res_id_int}",
                'Target_Chain': target_chain_proximity,
                'Query_Atom': neighbor['query_atom'],
                'Neighbor_Residue': f"{neighbor['neighbor_res_name']} {neighbor['neighbor_res_id']}",
                'Neighbor_Chain': neighbor['neighbor_chain_id'],
                'Neighbor_Atom': neighbor['neighbor_atom_name'],
                'Distance_Angstroms': neighbor['distance']
            })
    else:
        print(f"  No nearest neighboring atoms found within {cutoff} Å of {res_name_str} {res_id_int} (Chain {target_chain_proximity}) in the MODIFIED structure.")

# Create and display DataFrame for modified structure results
if all_proximity_results_modified:
    proximity_df_modified = pd.DataFrame(all_proximity_results_modified)
    print("\n--- Proximity Analysis Results DataFrame for Modified PTM Structure ---")
    display(proximity_df_modified)
    # Save to CSV
    proximity_output_csv_filename_modified = 'proximity_analysis_results_modified_ptm_new_targets.csv'
    proximity_df_modified.to_csv(proximity_output_csv_filename_modified, index=False)
    print(f"Proximity analysis results for modified PTM structure saved to {proximity_output_csv_filename_modified}")
else:
    print("\nNo proximity results to display for the modified PTM structure.")

In [ ]:
print("Combining Proximity Analysis Results for Comparison...")

# Ensure DataFrames are available (they should be from the previous cell execution)
if 'proximity_df_native' not in locals() or 'proximity_df_modified' not in locals():
    # If not in memory, load from CSVs
    try:
        proximity_df_native = pd.read_csv('proximity_analysis_results_native_new_targets.csv')
        proximity_df_modified = pd.read_csv('proximity_analysis_results_modified_ptm_new_targets.csv')
        print("Loaded proximity results from CSV files.")
    except FileNotFoundError:
        print("Error: Proximity analysis CSV files not found. Please run the previous proximity analysis cell.")
        # exit(1) # Or handle this error gracefully

# Add a 'Structure_Type' column to each DataFrame for easy identification
if 'proximity_df_native' in locals():
    proximity_df_native['Structure_Type'] = 'Native'
if 'proximity_df_modified' in locals():
    proximity_df_modified['Structure_Type'] = 'Modified'

# Combine the two DataFrames
if 'proximity_df_native' in locals() and 'proximity_df_modified' in locals():
    combined_proximity_df = pd.concat([proximity_df_native, proximity_df_modified], ignore_index=True)
    print("\nCombined Proximity Results:")
    display(combined_proximity_df)

    # Optionally, analyze differences
    # Merge on relevant columns to find common neighbors and compare distances
    # This assumes 'Query_Atom', 'Neighbor_Residue', 'Neighbor_Chain', 'Neighbor_Atom' uniquely identify a neighbor interaction
    merged_comparison_df = pd.merge(
        proximity_df_native.drop(columns=['Structure_Type']),
        proximity_df_modified.drop(columns=['Structure_Type']),
        on=['Target_Residue', 'Target_Chain', 'Query_Atom', 'Neighbor_Residue', 'Neighbor_Chain', 'Neighbor_Atom'],
        suffixes=('_Native', '_Modified'),
        how='outer'
    )

    # Calculate change in distance, handle NaNs for new/removed neighbors
    merged_comparison_df['Distance_Change'] = merged_comparison_df['Distance_Angstroms_Modified'] - merged_comparison_df['Distance_Angstroms_Native']

    print("\nProximity Comparison (Native vs. Modified):")
    display(merged_comparison_df.sort_values(by=['Target_Residue', 'Distance_Angstroms_Native']))

    # Save the combined and comparison results
    combined_proximity_output_csv = 'combined_proximity_analysis.csv'
    merged_comparison_output_csv = 'proximity_comparison_analysis.csv'

    combined_proximity_df.to_csv(combined_proximity_output_csv, index=False)
    merged_comparison_df.to_csv(merged_comparison_output_csv, index=False)

    print(f"Combined proximity results saved to {combined_proximity_output_csv}")
    print(f"Proximity comparison results saved to {merged_comparison_output_csv}")
else:
    print("Proximity DataFrames not available for combination or comparison.")